In [1]:
import os
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class UTKFaceDataset(Dataset):
    def __init__(self, filenames, data_dir, img_size=128):
        self.filenames = filenames
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        parts = filename.split('_')
        
        age = float(parts[0])
        gender = float(parts[1]) # 0 = Male, 1 = Female
        
        img_path = os.path.join(self.data_dir, filename)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (self.img_size, self.img_size))
        
        # PyTorch expects dimensions as (Channels, Height, Width)
        img = img.transpose((2, 0, 1)) / 255.0
        
        return (torch.tensor(img, dtype=torch.float32), 
                torch.tensor(age, dtype=torch.float32), 
                torch.tensor(gender, dtype=torch.float32))

# 1. Target your folder path
DATA_DIR = "archive (1)/UTKFace"
all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.jpg') and len(f.split('_')) >= 3]

# 2. Split data: 80% Train, 20% Test
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

# 3. Build DataLoaders
train_dataset = UTKFaceDataset(train_files, DATA_DIR)
test_dataset = UTKFaceDataset(test_files, DATA_DIR)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Dataset Loaded Successfully!")
print(f"Total training batches: {len(train_loader)}")
print(f"Total testing batches: {len(test_loader)}")

Dataset Loaded Successfully!
Total training batches: 593
Total testing batches: 149


In [2]:
import torch
print("Is CUDA available?", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Define the device strategy
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Is CUDA available? True
GPU Name: NVIDIA GeForce RTX 3050 Laptop GPU
Using device: cuda


In [3]:
import torch.nn as nn

class AgeGenderModel(nn.Module):
    def __init__(self):
        super(AgeGenderModel, self).__init__()
        
        # Convolutional Base (Extracting visual features like wrinkles, jawline, etc.)
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Resizes 128x128 -> 64x64
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Resizes 64x64 -> 32x32
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # Resizes 32x32 -> 16x16
            
            # Block 4
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # Resizes 16x16 -> 8x8
        )
        
        # Fully Connected Layers (The "Brain" making sense of the features)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5), # Prevents memorizing the data
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Dual Outputs
        self.age_output = nn.Linear(128, 1)        # Predicts a continuous number (Age)
        self.gender_output = nn.Linear(128, 1)     # Predicts 0 or 1 (Gender)

    def forward(self, x):
        x = self.features(x)
        x = self.fc(x)
        age = self.age_output(x)
        gender = torch.sigmoid(self.gender_output(x)) # Squeezes gender output between 0 and 1
        return age.squeeze(), gender.squeeze()

# Set device to your RTX 3050 GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model and send it to the GPU
model = AgeGenderModel().to(device)
print(f"Model initialized and moved to {device} successfully!")

Model initialized and moved to cuda successfully!


In [ ]:
import torch.optim as optim

# 1. Define how we calculate errors (Loss Functions)
criterion_age = nn.L1Loss()     # Mean Absolute Error (e.g., "off by 4 years")
criterion_gender = nn.BCELoss() # Binary Cross Entropy (Standard for 0/1 classification)

# 2. Define the Optimizer (The algorithm that updates the weights)
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 15
print("Training started on your RTX 3050... Watch the average loss drop!")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for i, (imgs, ages, genders) in enumerate(train_loader):
        # Move data to GPU
        imgs, ages, genders = imgs.to(device), ages.to(device), genders.to(device)
        
        # Clear old gradients
        optimizer.zero_grad()
        
        # Forward pass: guess the age and gender
        pred_age, pred_gender = model(imgs)
        
        # Calculate how wrong the guesses were
        loss_age = criterion_age(pred_age, ages)
        loss_gender = criterion_gender(pred_gender, genders)
        
        # Combine losses (We multiply gender by 10 so the model pays equal attention to it)
        total_loss = loss_age + (loss_gender * 10)
        
        # Backward pass: calculate the updates
        total_loss.backward()
        
        # Update the weights
        optimizer.step()
        
        running_loss += total_loss.item()
        
        # Print a tiny progress update every 100 batches so you know it hasn't frozen
        if (i + 1) % 100 == 0:
            print(f"  Batch {i+1}/{len(train_loader)} processed...")
            
    # Print the summary at the end of each epoch
    avg_epoch_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] Completed - Average Loss: {avg_epoch_loss:.4f}")
    print("-" * 50)

# Save your trained model weights to disk so you can use it later for the webcam script!
torch.save(model.state_dict(), 'best_age_gender_model.pth')
print("Training complete! Weights saved successfully as 'best_age_gender_model.pth'")


Training started on your RTX 3050... Watch the average loss drop!
  Batch 100/593 processed...
  Batch 200/593 processed...
  Batch 300/593 processed...
  Batch 400/593 processed...
  Batch 500/593 processed...
Epoch [1/15] Completed - Average Loss: 20.0959
--------------------------------------------------
  Batch 100/593 processed...
  Batch 200/593 processed...
  Batch 300/593 processed...
  Batch 400/593 processed...
  Batch 500/593 processed...
Epoch [2/15] Completed - Average Loss: 15.6278
--------------------------------------------------
  Batch 100/593 processed...
  Batch 200/593 processed...
  Batch 300/593 processed...
  Batch 400/593 processed...
  Batch 500/593 processed...
Epoch [3/15] Completed - Average Loss: 13.8747
--------------------------------------------------
  Batch 100/593 processed...
  Batch 200/593 processed...
  Batch 300/593 processed...
  Batch 400/593 processed...
  Batch 500/593 processed...
Epoch [4/15] Completed - Average Loss: 12.9754
-------------